# Loss Landscape Investigation

Load trained model and analyze with loss landscape inspection.
Uses context-controlled execution to generate datasets and load the model.

In [ ]:
from pathlib import Path

log_directory = Path(
    ""
)

## Step 1: Initialize Execution Context

Create the execution context that will be used to manage configuration and dataset generation.

In [ ]:
from frame.context.execution_context import ExecutionContext
from frame.file_structure import CONTEXT_FILE_NAME
from train.train_config import TrainConfig
from data_tools.detector.detector_config import DetectorConfig

if not log_directory.exists():
    raise ValueError(f"Selected directory does not exist: {log_directory}")

# Load context from the chosen directory
context_file = log_directory / CONTEXT_FILE_NAME
if not context_file.exists():
    raise FileNotFoundError(f"Context file not found: {context_file}")

context = ExecutionContext.naive_load_from_file(context_file)

print(f"✓ Context loaded from: {log_directory}")
print(f"✓ Config type: {type(context.config).__name__}")

# Type casting safety checks
config = context.config
if not isinstance(config, TrainConfig):
    raise TypeError(f"Expected TrainConfig, got {config.__class__.__name__}")
if not isinstance(config, DetectorConfig):
    raise TypeError(f"Expected DetectorConfig, got {config.__class__.__name__}")

print("✓ Execution context initialized successfully")

## Step 2: Generate Datasets

Create datasets for TauMuon decay process and apply detector effects.

In [ ]:
from data_tools.data_generation import DataGeneration

print("Generating datasets...")
gen = DataGeneration(context)
A_dataset, A_params = gen["TauMuon"]
B_dataset, B_params = gen["TauElectron"]

print("✓ Datasets generated successfully")
print(f"  A_dataset shape: {A_dataset.events.shape}")
print(f"  B_dataset shape: {B_dataset.events.shape}")

In [ ]:
from data_tools.detector.detector_effect import DetectorEffect

# Simulate detector effects
print("Applying detector effects...")
det = DetectorEffect(context)
detected_A_dataset = det.affect_and_compensate(A_dataset, A_params, is_display=False)
detected_B_dataset = det.affect_and_compensate(B_dataset, B_params, is_display=False)
reference_dataset = detected_A_dataset + detected_B_dataset

print("✓ Detector effects applied successfully")
print(f"  Detected A_dataset shape: {detected_A_dataset.events.shape}")
print(f"  Detected B_dataset shape: {detected_B_dataset.events.shape}")
print(f"  Reference dataset shape: {reference_dataset.events.shape}")

## Step 3: Load Trained Model

Locate and load the checkpoint from training results.

In [ ]:
import torch
from pathlib import Path
from neural_networks.differentiating_model import DifferentiatingModel

# Find checkpoint
checkpoint_path = next( Path(log_directory).rglob("**/*.ckpt"), None)

print(f"Loading model from: {checkpoint_path}")

# Create model instance
model_a = DifferentiatingModel(
    context=context,
    detector_effect=det,
    name="A_model_loaded",
)

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=model_a.device, weights_only=False)
model_a.load_state_dict(checkpoint['state_dict'])
model_a.eval()

print("✓ Model loaded successfully")
print(f"✓ Model device: {model_a.device}")

## Step 4: Prepare Test Batch

Create feature dataset and targets for analysis.

In [ ]:
import numpy as np

feature_dataset = detected_A_dataset + reference_dataset
target_structure = np.concatenate((
    np.ones(shape=(detected_A_dataset.n_samples,)),
    np.zeros(shape=(reference_dataset.n_samples,)),
), axis=0)
loss_weights = np.concatenate((
    detected_A_dataset._weight_mask,
    reference_dataset._weight_mask * detected_A_dataset.corrected_n_samples / reference_dataset.corrected_n_samples,
), axis=0)

print(f"Dataset info:")
print(f"  Feature dimension: {feature_dataset.events.shape[1]}")
print(f"  Total samples: {feature_dataset.n_samples}")
print(f"  Targets: {target_structure.shape}")
print(f"  Weights: {loss_weights.shape}")

## Step 5: Compute Loss on Test Batch

Evaluate model loss using the ddp_symmetrized_loss function.

In [ ]:
import torch

# Prepare test batch (first 100 samples)
x_batch = torch.tensor(feature_dataset.events[:100], dtype=torch.float32, device=model_a.device)
y_batch = torch.tensor(target_structure[:100], dtype=torch.float32, device=model_a.device)
w_batch = torch.tensor(loss_weights[:100], dtype=torch.float32, device=model_a.device)

# Compute loss
with torch.no_grad():
    predictions = model_a(x_batch, training=False)
    batch_loss = model_a.ddp_symmetrized_loss(y_batch, predictions)
    weighted_loss = (batch_loss * w_batch).mean()

print(f"Test batch loss (ddp_symmetrized_loss):")
print(f"  Loss shape: {batch_loss.shape}")
print(f"  Loss mean: {batch_loss.mean().item():.6f}")
print(f"  Weighted loss (mean): {weighted_loss.item():.6f}")
print(f"  Predictions shape: {predictions.shape}")
print(f"  Predictions range: [{predictions.min().item():.4f}, {predictions.max().item():.4f}]")

## Step 6: Model Architecture Summary

Display information about the loaded model architecture and parameters.

In [ ]:
total_params = sum(p.numel() for p in model_a.parameters())
trainable_params = sum(p.numel() for p in model_a.parameters() if p.requires_grad)

print(f"Model architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Input dimension: {config.train__nn_input_dimension}")
print(f"  Hidden dimension: {config.train__nn_inner_layer_nodes}")
print(f"  Output dimension: {config.train__nn_output_dimension}")

print("\n✓ Model ready for analysis with loss landscape utilities!")

## Step 7: Loss Landscape Analysis

Compute and visualize the loss landscape to understand the geometry of the solution found by the model.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from typing import Callable, Tuple, Optional

# Loss landscape utilities
def compute_1d_loss_slice(
    model: torch.nn.Module,
    loss_fn: Callable,
    data_batch: Tuple[torch.Tensor, torch.Tensor],
    direction: list,
    alphas: np.ndarray = np.linspace(-1, 1, 50),
    device: str = 'cpu',
) -> np.ndarray:
    """Compute loss along a 1D direction in parameter space."""
    x, y = data_batch
    x, y = x.to(device), y.to(device)
    
    # Store original parameters
    original_params = [p.clone().detach() for p in model.parameters()]
    
    losses = []
    model.eval()
    
    with torch.no_grad():
        for alpha in alphas:
            # Perturb parameters along direction
            param_idx = 0
            for p in model.parameters():
                p.data = original_params[param_idx] + alpha * direction[param_idx]
                param_idx += 1
            
            # Compute loss
            output = model(x)
            loss = loss_fn(output, y)
            # Handle both scalar and vector losses
            if loss.numel() == 1:
                loss_val = loss.item()
            else:
                loss_val = loss.mean().item()
            losses.append(loss_val)
    
    # Restore original parameters
    param_idx = 0
    for p in model.parameters():
        p.data = original_params[param_idx]
        param_idx += 1
    
    return np.array(losses)


def compute_2d_loss_surface(
    model: torch.nn.Module,
    loss_fn: Callable,
    data_batch: Tuple[torch.Tensor, torch.Tensor],
    direction1: list,
    direction2: list,
    alphas: np.ndarray = np.linspace(-1, 1, 30),
    betas: np.ndarray = np.linspace(-1, 1, 30),
    device: str = 'cpu',
) -> np.ndarray:
    """Compute 2D loss surface by varying parameters in two directions."""
    x, y = data_batch
    x, y = x.to(device), y.to(device)
    
    original_params = [p.clone().detach() for p in model.parameters()]
    loss_surface = np.zeros((len(alphas), len(betas)))
    
    model.eval()
    
    with torch.no_grad():
        for i, alpha in enumerate(alphas):
            for j, beta in enumerate(betas):
                # Perturb in both directions
                param_idx = 0
                for p in model.parameters():
                    p.data = original_params[param_idx] + alpha * direction1[param_idx] + beta * direction2[param_idx]
                    param_idx += 1
                
                # Compute loss
                output = model(x)
                loss = loss_fn(output, y)
                # Handle both scalar and vector losses
                if loss.numel() == 1:
                    loss_val = loss.item()
                else:
                    loss_val = loss.mean().item()
                loss_surface[i, j] = loss_val
    
    # Restore
    param_idx = 0
    for p in model.parameters():
        p.data = original_params[param_idx]
        param_idx += 1
    
    return loss_surface


def generate_random_direction(model: torch.nn.Module) -> list:
    """Generate random direction in parameter space."""
    direction = []
    for p in model.parameters():
        direction.append(torch.randn_like(p))
    return direction


def normalize_direction(direction: list) -> list:
    """Normalize direction vector."""
    norm = np.sqrt(sum(torch.sum(d ** 2).item() for d in direction))
    return [d / (norm + 1e-10) for d in direction]


print("✓ Loss landscape utilities defined")

In [ ]:
def plot_1d_loss_slice(
    losses: np.ndarray,
    alphas: np.ndarray,
    title: str = "1D Loss Slice",
    save_path: Optional[Path] = None,
) -> plt.Figure:
    """Plot 1D loss profile."""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(alphas, losses, 'b-', linewidth=2, label='Loss')
    ax.axvline(0, color='r', linestyle='--', label='Current parameters')
    ax.fill_between(alphas, losses, alpha=0.3)
    
    ax.set_xlabel('Step size (α)', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(title, fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add curvature indicator
    curvature = np.gradient(np.gradient(losses))
    avg_curvature = np.mean(np.abs(curvature))
    ax.text(0.02, 0.98, f'Avg Curvature: {avg_curvature:.4f}', 
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    
    return fig


def plot_2d_loss_surface(
    loss_surface: np.ndarray,
    alphas: np.ndarray,
    betas: np.ndarray,
    title: str = "2D Loss Surface",
    save_path: Optional[Path] = None,
) -> plt.Figure:
    """Plot 2D loss surface as heatmap."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Heatmap
    im = ax1.contourf(betas, alphas, loss_surface, levels=20, cmap='viridis')
    ax1.plot(0, 0, 'r*', markersize=15, label='Current parameters')
    ax1.set_xlabel('Direction 2 (β)', fontsize=12)
    ax1.set_ylabel('Direction 1 (α)', fontsize=12)
    ax1.set_title(f'{title} (Heatmap)', fontsize=12)
    ax1.legend()
    cbar = plt.colorbar(im, ax=ax1)
    cbar.set_label('Loss', fontsize=10)
    
    # Contour plot
    levels = np.linspace(loss_surface.min(), loss_surface.max(), 20)
    cs = ax2.contour(betas, alphas, loss_surface, levels=levels)
    ax2.clabel(cs, inline=True, fontsize=8)
    ax2.plot(0, 0, 'r*', markersize=15, label='Current parameters')
    ax2.set_xlabel('Direction 2 (β)', fontsize=12)
    ax2.set_ylabel('Direction 1 (α)', fontsize=12)
    ax2.set_title(f'{title} (Contours)', fontsize=12)
    ax2.legend()
    
    # Calculate basin width (flat minimum indicator)
    center_loss = loss_surface[len(alphas)//2, len(betas)//2]
    threshold = center_loss * 1.1  # 10% above center
    flat_region = np.sum(loss_surface < threshold)
    total_region = loss_surface.size
    flatness_ratio = flat_region / total_region
    
    fig.suptitle(f'{title}\nFlatness ratio: {flatness_ratio:.1%}', fontsize=14)
    
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    
    return fig


print("✓ Plotting utilities defined")

### 1D Loss Slice Analysis

Analyze how the loss varies along a random direction in parameter space.

In [ ]:
from pathlib import Path

print("Computing 1D loss slice...")
direction1 = generate_random_direction(model_a)
direction1 = normalize_direction(direction1)

alphas = np.linspace(-1, 1, 50)
losses_1d = compute_1d_loss_slice(
    model_a,
    lambda y_pred, y_true: model_a.ddp_symmetrized_loss(y_true, y_pred),
    (x_batch, y_batch),
    direction1,
    alphas=alphas,
    device=model_a.device,
)

print(f"✓ 1D loss slice computed")
print(f"  Loss at origin: {losses_1d[len(losses_1d)//2]:.6f}")
print(f"  Min loss: {np.min(losses_1d):.6f}")
print(f"  Max loss: {np.max(losses_1d):.6f}")
print(f"  Loss range: {np.max(losses_1d) - np.min(losses_1d):.6f}")

# Curvature analysis
curvature_1d = np.gradient(np.gradient(losses_1d))
sharpness_1d = np.std(np.gradient(losses_1d))
avg_curvature_1d = np.mean(np.abs(curvature_1d))

print(f"  Average curvature: {avg_curvature_1d:.6f}")
print(f"  Gradient sharpness: {sharpness_1d:.6f}")

# Plot
fig_1d = plot_1d_loss_slice(losses_1d, alphas, title="1D Loss Slice - Random Direction")
plt.show()

### 2D Loss Surface Analysis

Analyze the full 2D loss surface around the current solution.

In [ ]:
print("Computing 2D loss surface...")
direction2 = generate_random_direction(model_a)
direction2 = normalize_direction(direction2)

alphas_2d = np.linspace(-1, 1, 30)
betas_2d = np.linspace(-1, 1, 30)

loss_surface = compute_2d_loss_surface(
    model_a,
    lambda y_pred, y_true: model_a.ddp_symmetrized_loss(y_true, y_pred),
    (x_batch, y_batch),
    direction1,
    direction2,
    alphas=alphas_2d,
    betas=betas_2d,
    device=model_a.device,
)

print(f"✓ 2D loss surface computed")
center_loss = loss_surface[len(alphas_2d)//2, len(betas_2d)//2]
print(f"  Loss at center: {center_loss:.6f}")
print(f"  Min loss: {np.min(loss_surface):.6f}")
print(f"  Max loss: {np.max(loss_surface):.6f}")
print(f"  Basin depth: {np.max(loss_surface) - center_loss:.6f}")

# Flatness analysis
threshold = center_loss * 1.1  # 10% above center
flat_region = np.sum(loss_surface < threshold)
total_region = loss_surface.size
flatness_ratio = flat_region / total_region
print(f"  Flatness ratio (10% threshold): {flatness_ratio:.1%}")

# Plot
fig_2d = plot_2d_loss_surface(loss_surface, alphas_2d, betas_2d, 
                               title="2D Loss Surface - Random Directions")
plt.show()

## Step 8: Landscape Summary

Summarize key findings from the loss landscape analysis.

In [ ]:
import matplotlib.pyplot as plt

# Generate summary report
summary = {
    "model_name": model_a._name,
    "total_parameters": total_params,
    "trainable_parameters": trainable_params,
    "batch_size": x_batch.shape[0],
    "feature_dimension": feature_dataset.events.shape[1],
    "test_loss": batch_loss.mean().item(),
    "weighted_loss": weighted_loss.item(),
    "loss_landscape": {
        "1d_curvature": avg_curvature_1d,
        "1d_sharpness": sharpness_1d,
        "1d_loss_range": np.max(losses_1d) - np.min(losses_1d),
        "2d_flatness_ratio": flatness_ratio,
        "2d_basin_depth": np.max(loss_surface) - center_loss,
    }
}

print("\n" + "="*60)
print("LOSS LANDSCAPE ANALYSIS SUMMARY")
print("="*60)
print(f"\nModel: {summary['model_name']}")
print(f"Parameters: {summary['trainable_parameters']:,} trainable / {summary['total_parameters']:,} total")
print(f"Batch: {summary['batch_size']} samples, {summary['feature_dimension']} features")

print(f"\n--- Loss Metrics ---")
print(f"Test loss (mean): {summary['test_loss']:.6f}")
print(f"Weighted loss: {summary['weighted_loss']:.6f}")

print(f"\n--- 1D Landscape (Random Direction) ---")
print(f"Curvature: {summary['loss_landscape']['1d_curvature']:.6f}")
print(f"Gradient sharpness: {summary['loss_landscape']['1d_sharpness']:.6f}")
print(f"Loss range: {summary['loss_landscape']['1d_loss_range']:.6f}")
if summary['loss_landscape']['1d_curvature'] < 0.01:
    print("  → FLAT: Good for exploration/generalization")
elif summary['loss_landscape']['1d_curvature'] > 0.1:
    print("  → SHARP: Model may be overfitting")
else:
    print("  → MODERATE: Balanced solution")

print(f"\n--- 2D Landscape (Two Random Directions) ---")
print(f"Flatness ratio (10% threshold): {summary['loss_landscape']['2d_flatness_ratio']:.1%}")
print(f"Basin depth: {summary['loss_landscape']['2d_basin_depth']:.6f}")
if summary['loss_landscape']['2d_flatness_ratio'] > 0.5:
    print("  → BROAD BASIN: Robust solution")
else:
    print("  → SHARP BASIN: Narrow minimum, sensitive to perturbations")

print("\n--- Interpretation ---")
if summary['loss_landscape']['1d_curvature'] < 0.01 and summary['loss_landscape']['2d_flatness_ratio'] > 0.5:
    print("✓ Model has found a FLAT, ROBUST minimum")
    print("  Good properties for generalization and robustness")
elif summary['loss_landscape']['1d_curvature'] > 0.1 or summary['loss_landscape']['2d_flatness_ratio'] < 0.3:
    print("⚠ Model is at a SHARP, NARROW minimum")
    print("  May indicate overfitting or limited exploration")
else:
    print("• Model is at a MIXED minimum")
    print("  Moderate curvature and flatness characteristics")

print("="*60 + "\n")